<div style="background-color: steelblue; padding: 10px; border-radius: 5px;">
    <p style="margin: 10;"></p>
    <h1 style="text-align: center; margin: 0; font-weight: bold; color: white;">SWOT OMIP : Global estimation of spectrum</h1>
    <p style="margin: 10;"></p>
</div>



## 📦**Imports** 

In [1]:
%%time

# -----------------------------
# Imports
# -----------------------------

from datetime import datetime
import s3fs
import xarray as xr
import pyinterp
import numpy as np
import json
from watermark import watermark
import time
import platform
from shapely.geometry import shape, box
from shapely import geometry
import geopandas as gpd
from shapely import geometry

import geopandas as gpd
import save_spect_json2 as ssj

import DataPreprocessing_listv2 as dp #Modified from widetrax % I should create widetrax branch
from widetrax import Spectra as sp
#import importlib
#importlib.reload(dp)


CPU times: user 1.65 s, sys: 513 ms, total: 2.17 s
Wall time: 5.83 s


## 🧮 **Required variables**

In [2]:

# -----------------------------
# Define variables
# -----------------------------

season="ASO" # Initial of the season
prefix='GLO12V4' # Models
s3_folder =f"s3://project-moi-swot-omip/{prefix}" # this depend of the information shared for contributors
endpoint_url =  "https://minio.dive.edito.eu/" # this depend of the information shared for contributors


## 🔍 **Check the S3 Endpoint**  

In [3]:
# -----------------------------
# Verify that data is available
# -----------------------------
fs = s3fs.S3FileSystem(anon=True,endpoint_url=endpoint_url)
# List contents of the bucket
contents = fs.ls(s3_folder)
print("Bucket contents:")
for item in contents:
    print(item)

Bucket contents:
project-moi-swot-omip/GLO12V4/cycle_
project-moi-swot-omip/GLO12V4/cycle_008
project-moi-swot-omip/GLO12V4/cycle_009
project-moi-swot-omip/GLO12V4/cycle_010
project-moi-swot-omip/GLO12V4/cycle_011
project-moi-swot-omip/GLO12V4/cycle_012
project-moi-swot-omip/GLO12V4/cycle_013
project-moi-swot-omip/GLO12V4/cycle_014
project-moi-swot-omip/GLO12V4/cycle_015
project-moi-swot-omip/GLO12V4/cycle_016
project-moi-swot-omip/GLO12V4/cycle_017
project-moi-swot-omip/GLO12V4/cycle_018
project-moi-swot-omip/GLO12V4/cycle_019
project-moi-swot-omip/GLO12V4/cycle_020
project-moi-swot-omip/GLO12V4/cycle_021
project-moi-swot-omip/GLO12V4/cycle_022
project-moi-swot-omip/GLO12V4/cycle_023
project-moi-swot-omip/GLO12V4/cycle_024
project-moi-swot-omip/GLO12V4/cycle_025
project-moi-swot-omip/GLO12V4/cycle_026


## 🔄 **Identify the cycle numbers within the specified time range** 

In [4]:
# -----------------------------
# Find cycle coincident with the seasonality
# Nothing to do here (unless you want to add some season)
# -----------------------------
if season=="JFM":    
    start_date = "01012024" # "DDMMYYYY"
    end_date ="31032024"
elif season=="FMA":
    start_date = "01022024" # "DDMMYYYY"
    end_date ="30042024"
elif season=="AMJ":
    start_date = "01042024" # "DDMMYYYY"
    end_date ="30062024"
elif season=="JAS":
    start_date = "01072024" # "DDMMYYYY"
    end_date ="30092024"
elif season=="ASO":
    start_date = "01082024" # "DDMMYYYY"
    end_date ="31102024"
elif season=="OND":
    start_date = "01102024" # "DDMMYYYY"
    end_date ="31122024"

if season =="JFM":
    file_path = "https://minio.lab.dive.edito.eu/project-meom-ige/cycles_periods.csv" # works only for winter period
#elif season =="JAS":
else:
    file_path = "time_ranges.csv"  # for summer
matching_cycles = dp.get_matching_cycles(file_path, start_date, end_date)

def formater_numeros_concis(liste_numeros):
  return [str(numero).zfill(3) for numero in liste_numeros]
    
matching_cycles = formater_numeros_concis(matching_cycles)
matching_cycles

['019', '020', '021', '022', '023']

##  **Estimation of SSH spectra for every boxes over the global ocean** 

In [5]:
%%time

# -----------------------------
# Load boxes position from json files
# -----------------------------
with open("mostly_ocean_boxes_filtered.geojson") as f:
    data = json.load(f)

# -----------------------------
# Loop for the estimation for every box
# -----------------------------
bx=0
for feature in data["features"][:]:
    # Load position for one specific box
    poly = shape(feature["geometry"])
    specific_area = poly
    #lon_min, lat_min, lon_max, lat_max
    area = list(poly.bounds)
    area[0]=(area[0] + 360) % 360
    area[2]=(area[2] + 360) % 360
    area = tuple(area)
    print(area)
    lon_min, lat_min, lon_max, lat_max = area

    #  Finding the track for specific time and space
    phase = 'science'
    GEOMETRIES_FILE = f'KaRIn_2kms_{phase}_geometries.geojson'
    swath_geometries = gpd.read_file(GEOMETRIES_FILE)
    swath_geometries['geometry'] = swath_geometries.geometry.apply(dp.normalize_longitude)
    swath_geometries.crs = 'EPSG:4326'  # Ensure WGS84
    matching_polys = swath_geometries[swath_geometries.intersects(specific_area)]
    pass_numbers = list(matching_polys['pass_number'])
    list_files=dp.find_listdata(matching_cycles, endpoint_url,s3_folder, pass_numbers, start_date, end_date)
    datasets_dict = dp.read_swot_ncfiles_S3folder_list(s3_folder, endpoint_url, area, list_files,engine="h5netcdf")
    has_converged, filled_datasets = dp.fill_nan(datasets_dict, varname = "ssh")
    segments_dict = sp.retrieve_segments(filled_datasets,FileType = "NetCDF",namevar="ssh")

    # Calculate PSD 
    psd_dict, freqs_dict = sp.calculate_psd(segments_dict)
    # Calculate PSD Mean
    psd_mean, freqs_mean = sp.psd_mean_and_freq(psd_dict,freqs_dict)

    ssj.save_spect((f"GLOBALv2/{prefix}/Global_box_"+str(bx)),f"{prefix}",season,freqs_mean,psd_mean,area[0],area[2],area[1],area[3],start_date,end_date)
    bx=bx+1


(200.0, -70.0, 210.0, -60.0)
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_039_20240801T034011_20240801T043138_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_041_20240801T052305_20240801T061431_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_056_20240801T181447_20240801T190613_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_058_20240801T195741_20240801T204907_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_067_20240802T034042_20240802T043209_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_069_20240802T052336_20240802T061502_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_084_20240802T181518_20240802T190644_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_086_20240802T195811_20240802T204938_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_095_20240803T034113_2024080

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(210.0, -70.0, 220.0, -60.0)
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_039_20240801T034011_20240801T043138_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_041_20240801T052305_20240801T061431_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_056_20240801T181447_20240801T190613_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_058_20240801T195741_20240801T204907_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_065_20240802T015748_20240802T024915_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_069_20240802T052336_20240802T061502_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_084_20240802T181518_20240802T190644_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_086_20240802T195811_20240802T204938_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_093_20240

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(220.0, -70.0, 230.0, -60.0)
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_037_20240801T015717_20240801T024844_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_041_20240801T052305_20240801T061431_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_056_20240801T181447_20240801T190613_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_058_20240801T195741_20240801T204907_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_065_20240802T015748_20240802T024915_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_069_20240802T052336_20240802T061502_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_082_20240802T163224_20240802T172351_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_086_20240802T195811_20240802T204938_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_093_20240

...

JSON file created
(300.0, 30.0, 310.0, 40.0)
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_046_20240801T094019_20240801T103145_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_056_20240801T181447_20240801T190613_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_061_20240801T223201_20240801T232327_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_071_20240802T070629_20240802T075756_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_074_20240802T094050_20240802T103216_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_084_20240802T181518_20240802T190644_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_089_20240802T223232_20240802T232358_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_102_20240803T094121_20240803T103247_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_117_2024080

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(310.0, 30.0, 320.0, 40.0)
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_046_20240801T094019_20240801T103145_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_056_20240801T181447_20240801T190613_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_061_20240801T223201_20240801T232327_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_071_20240802T070629_20240802T075756_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_074_20240802T094050_20240802T103216_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_084_20240802T181518_20240802T190644_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_227_20240807T205214_20240807T214340_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_240_20240808T080103_20240808T085229_v1.0.2.nc
project-moi-swot-omip/GLO12V4/cycle_019/SWOT_GRID_L3_LR_SSH_019_255_2024080

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
CPU times: user 1h 2min 3s, sys: 59min 58s, total: 2h 2min 2s
Wall time: 5h 44min 16s


## 💾 **Save Results and Information in JSON File**

## 📤 **Export Results to the S3 Endpoint** 

In [14]:
model=prefix
fs = s3fs.S3FileSystem( anon=True, endpoint_url="https://minio.lab.dive.edito.eu", use_ssl=False ) 
json_files = glob.glob(f"{model}/*.json")

for f in json_files:
    fs.put(f, f"project-meom-ige/OMIP/GLOBAL/{model}/{f.split('/')[-1]}")

[None]